I previously shared a [notebook](url) (see the discussion [here](https://www.kaggle.com/competitions/kaggle-llm-science-exam/discussion/441128)) that found a cluster of relevant Wikipedia STEM articles, resulting in around 270K STEM articles for which the resulting dataset is released [here.](https://www.kaggle.com/datasets/mbanaei/stem-wiki-cohere-no-emb)

However, due to issues with WikiExtractor, there're cases in which some numbers or even paragraphs are missing from the final Wiki parsing. Therefore,  for the same set of  articles, I used Wiki API to gather the articles' contexts (see discussion [here](https://www.kaggle.com/competitions/kaggle-llm-science-exam/discussion/442483)), for which the resulting dataset is released [here](https://www.kaggle.com/datasets/mbanaei/all-paraphs-parsed-expanded).

In order to show that the found articles cover not only the train dataset articles but also a majority of LB gold articles, I release this notebook that uses a simple retrieval model (without any prior indexing) together with a model that is trained only on the RACE dataset. (not fine-tuned on any competition-similar dataset).

The main design choices for the notebook are:
- Using a simple TF-IDF to retrieve contexts from both datasets for every given question.
- Although the majority of high-performing public models use DeBERTa-V3 to do the inference in their pipeline, I used a LongFormer Large model, which enables us to have a much longer prefix context given limited GPU memory. More specifically, as opposed to many public notebooks, there's no splitting to sentence level, and the whole paragraph is retrieved and passed to the classifier as a context (the main reason that we don't get OOM and also have relatively fast inference is that in LongFormer full attention is not computed as opposed to standard models like BERT).
- I use a fall-back model (based on a public notebook that uses an openbook approach and performs 81.5 on LB) that is used for prediction when there's low confidence in the main model's output for the top choice.

P.S: Although the model's performance is relatively good compared to other public notebooks, many design choices can be revised to improve both inference time and performance. (e.g., currently, context retrieval seems to be the inference bottleneck as no prior indexing is used).

In [1]:
!cp /kaggle/input/datasets-wheel/datasets-2.14.4-py3-none-any.whl /kaggle/working
!pip install  /kaggle/working/datasets-2.14.4-py3-none-any.whl
!cp /kaggle/input/backup-806/util_openbook.py .

Processing ./datasets-2.14.4-py3-none-any.whl
  Attempting uninstall: datasets
    Found existing installation: datasets 2.1.0
    Uninstalling datasets-2.1.0:
      Successfully uninstalled datasets-2.1.0


In [2]:
# installing offline dependencies
!pip install -U /kaggle/input/faiss-gpu-173-python310/faiss_gpu-1.7.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
!cp -rf /kaggle/input/sentence-transformers-222/sentence-transformers /kaggle/working/sentence-transformers
!pip install -U /kaggle/working/sentence-transformers
!pip install -U /kaggle/input/blingfire-018/blingfire-0.1.8-py3-none-any.whl

!pip install --no-index --no-deps /kaggle/input/llm-whls/transformers-4.31.0-py3-none-any.whl
!pip install --no-index --no-deps /kaggle/input/llm-whls/peft-0.4.0-py3-none-any.whl
!pip install --no-index --no-deps /kaggle/input/llm-whls/trl-0.5.0-py3-none-any.whl

Processing /kaggle/input/faiss-gpu-173-python310/faiss_gpu-1.7.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing ./sentence-transformers
  Preparing metadata (setup.py) ... - \ done
  Created wheel for sentence-transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=126134 sha256=14a244c3186425c965633e82a9947653c1971d644d80c50d22fed35cfc298e56
  Stored in directory: /root/.cache/pip/wheels/6c/ea/76/d9a930b223b1d3d5d6aff69458725316b0fe205b854faf1812
Successfully built sentence-transformers
Processing /kaggle/input/blingfire-018/blingfire-0.1.8-py3-none-any.whl
Processing /kaggle/input/llm-whls/transformers-4.31.0-py3-none-any.whl
  Attempting uninstall: transformers
    Found existing installation: transformers 4.30.2
    Uninstalling transformers-4.30.2:
      Successfully uninstalled transformers-4.30.2
Processing /kaggle/input/llm-whls/peft-0.4.0-py3-none-any.whl
Processing /kaggle/input/llm-whls/trl-0.5.0-py3-none-any.whl


In [3]:
from util_openbook import get_contexts, generate_openbook_output
import pickle

get_contexts()
generate_openbook_output()

import gc
gc.collect()

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: l

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/3545 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
You're using a DebertaV2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


22

In [4]:
import pandas as pd
backup_model_predictions = pd.read_csv("submission_backup.csv")

In [5]:
import numpy as np
import pandas as pd 
from datasets import load_dataset, load_from_disk
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
from transformers import LongformerTokenizer, LongformerForMultipleChoice
import transformers
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import unicodedata

import os

In [6]:
!cp -r /kaggle/input/stem-wiki-cohere-no-emb /kaggle/working
!cp -r /kaggle/input/all-paraphs-parsed-expanded /kaggle/working/

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [7]:
def SplitList(mylist, chunk_size):
    return [mylist[offs:offs+chunk_size] for offs in range(0, len(mylist), chunk_size)]

def get_relevant_documents_parsed(df_valid):
    df_chunk_size=600
    paraphs_parsed_dataset = load_from_disk("/kaggle/working/all-paraphs-parsed-expanded")
    modified_texts = paraphs_parsed_dataset.map(lambda example:
                                             {'temp_text':
                                              f"{example['title']} {example['section']} {example['text']}".replace('\n'," ").replace("'","")},
                                             num_proc=2)["temp_text"]
    
    all_articles_indices = []
    all_articles_values = []
    for idx in tqdm(range(0, df_valid.shape[0], df_chunk_size)):
        df_valid_ = df_valid.iloc[idx: idx+df_chunk_size]
    
        articles_indices, merged_top_scores = retrieval(df_valid_, modified_texts)
        all_articles_indices.append(articles_indices)
        all_articles_values.append(merged_top_scores)
        
    article_indices_array =  np.concatenate(all_articles_indices, axis=0)
    articles_values_array = np.concatenate(all_articles_values, axis=0).reshape(-1)
    
    top_per_query = article_indices_array.shape[1]
    articles_flatten = [(
                         articles_values_array[index],
                         paraphs_parsed_dataset[idx.item()]["title"],
                         paraphs_parsed_dataset[idx.item()]["text"],
                        )
                        for index,idx in enumerate(article_indices_array.reshape(-1))]
    retrieved_articles = SplitList(articles_flatten, top_per_query)
    return retrieved_articles



def get_relevant_documents(df_valid):
    df_chunk_size=800
    
    cohere_dataset_filtered = load_from_disk("/kaggle/working/stem-wiki-cohere-no-emb")
    modified_texts = cohere_dataset_filtered.map(lambda example:
                                             {'temp_text':
                                              unicodedata.normalize("NFKD", f"{example['title']} {example['text']}").replace('"',"")},
                                             num_proc=2)["temp_text"]
    
    all_articles_indices = []
    all_articles_values = []
    for idx in tqdm(range(0, df_valid.shape[0], df_chunk_size)):
        df_valid_ = df_valid.iloc[idx: idx+df_chunk_size]
    
        articles_indices, merged_top_scores = retrieval(df_valid_, modified_texts)
        all_articles_indices.append(articles_indices)
        all_articles_values.append(merged_top_scores)
        
    article_indices_array =  np.concatenate(all_articles_indices, axis=0)
    articles_values_array = np.concatenate(all_articles_values, axis=0).reshape(-1)
    
    top_per_query = article_indices_array.shape[1]
    articles_flatten = [(
                         articles_values_array[index],
                         cohere_dataset_filtered[idx.item()]["title"],
                         unicodedata.normalize("NFKD", cohere_dataset_filtered[idx.item()]["text"]),
                        )
                        for index,idx in enumerate(article_indices_array.reshape(-1))]
    retrieved_articles = SplitList(articles_flatten, top_per_query)
    return retrieved_articles



def retrieval(df_valid, modified_texts):
    
    corpus_df_valid = df_valid.apply(lambda row:
                                     f'{row["prompt"]}\n{row["prompt"]}\n{row["prompt"]}\n{row["A"]}\n{row["B"]}\n{row["C"]}\n{row["D"]}\n{row["E"]}',
                                     axis=1).values
    vectorizer1 = TfidfVectorizer(ngram_range=(1,3),
                                 token_pattern=r"(?u)\b[\w/.-]+\b|!|/|\?|\"|\'",
                                 stop_words=stop_words)
    vectorizer1.fit(corpus_df_valid)
    vocab_df_valid = vectorizer1.get_feature_names_out()
    vectorizer = TfidfVectorizer(ngram_range=(1,3),
                                 token_pattern=r"(?u)\b[\w/.-]+\b|!|/|\?|\"|\'",
                                 stop_words=stop_words,
                                 vocabulary=vocab_df_valid)
    vectorizer.fit(modified_texts[:500000])
    corpus_tf_idf = vectorizer.transform(corpus_df_valid)
    
    print(f"length of vectorizer vocab is {len(vectorizer.get_feature_names_out())}")

    chunk_size = 100000
    top_per_chunk = 10
    top_per_query = 10

    all_chunk_top_indices = []
    all_chunk_top_values = []

    for idx in tqdm(range(0, len(modified_texts), chunk_size)):
        wiki_vectors = vectorizer.transform(modified_texts[idx: idx+chunk_size])
        temp_scores = (corpus_tf_idf * wiki_vectors.T).toarray()
        chunk_top_indices = temp_scores.argpartition(-top_per_chunk, axis=1)[:, -top_per_chunk:]
        chunk_top_values = temp_scores[np.arange(temp_scores.shape[0])[:, np.newaxis], chunk_top_indices]

        all_chunk_top_indices.append(chunk_top_indices + idx)
        all_chunk_top_values.append(chunk_top_values)

    top_indices_array = np.concatenate(all_chunk_top_indices, axis=1)
    top_values_array = np.concatenate(all_chunk_top_values, axis=1)
    
    merged_top_scores = np.sort(top_values_array, axis=1)[:,-top_per_query:]
    merged_top_indices = top_values_array.argsort(axis=1)[:,-top_per_query:]
    articles_indices = top_indices_array[np.arange(top_indices_array.shape[0])[:, np.newaxis], merged_top_indices]
    
    return articles_indices, merged_top_scores


def prepare_answering_input(
        tokenizer, 
        question,  
        options,   
        context,   
        model,
        max_seq_length=4096,
    ):
    c_plus_q   = context + ' ' + tokenizer.bos_token + ' ' + question
    c_plus_q_4 = [c_plus_q] * len(options)
    tokenized_examples = tokenizer(
        c_plus_q_4, options,
        max_length=max_seq_length,
        padding="longest",
        truncation=False,
        return_tensors="pt",
    )
    input_ids = tokenized_examples['input_ids'].unsqueeze(0)
    attention_mask = tokenized_examples['attention_mask'].unsqueeze(0)
    example_encoded = {
        "input_ids": input_ids.to(model.device.index),
        "attention_mask": attention_mask.to(model.device.index),
    }
    return example_encoded


In [8]:
# stop_words = ['each', 'you', 'the', 'use', 'used',
#                   'where', 'themselves', 'nor', "it's", 'how', "don't", 'just', 'your',
#                   'about', 'himself', 'with', "weren't", 'hers', "wouldn't", 'more', 'its', 'were',
#                   'his', 'their', 'then', 'been', 'myself', 're', 'not',
#                   'ours', 'will', 'needn', 'which', 'here', 'hadn', 'it', 'our', 'there', 'than',
#                   'most', "couldn't", 'both', 'some', 'for', 'up', 'couldn', "that'll",
#                   "she's", 'over', 'this', 'now', 'until', 'these', 'few', 'haven',
#                   'of', 'wouldn', 'into', 'too', 'to', 'very', 'shan', 'before', 'the', 'they',
#                   'between', "doesn't", 'are', 'was', 'out', 'we', 'me',
#                   'after', 'has', "isn't", 'have', 'such', 'should', 'yourselves', 'or', 'during', 'herself',
#                   'doing', 'in', "shouldn't", "won't", 'when', 'do', 'through', 'she',
#                   'having', 'him', "haven't", 'against', 'itself', 'that',
#                   'did', 'theirs', 'can', 'those',
#                   'own', 'so', 'and', 'who', "you've", 'yourself', 'her', 'he', 'only',
#                   'what', 'ourselves', 'again', 'had', "you'd", 'is', 'other',
#                   'why', 'while', 'from', 'them', 'if', 'above', 'does', 'whom',
#                   'yours', 'but', 'being', "wasn't", 'be']


stop_words = ["don't",
 'did',
 'she',
 'shan',
 'am',
 'these',
 'isn',
 'use',
 'as',
 'but',
 'doesn',
 'will',
 'once',
 'after',
 "mustn't",
 'most',
 "isn't",
 'to',
 "that'll",
 'on',
 "needn't",
 'were',
 'other',
 'him',
 'some',
 'where',
 "you'd",
 'was',
 'of',
 'or',
 'any',
 'we',
 'my',
 'all',
 'under',
 'hasn',
 'you',
 "mightn't",
 "aren't",
 'his',
 'by',
 'just',
 'them',
 'wouldn',
 'itself',
 'up',
 "she's",
 'weren',
 'd',
 'here',
 'so',
 'while',
 'over',
 'off',
 'they',
 'each',
 'at',
 'own',
 'no',
 't',
 'this',
 'won',
 "shan't",
 'he',
 'having',
 'whom',
 'through',
 'too',
 'yourself',
 'between',
 'have',
 'll',
 'and',
 "you've",
 'until',
 'not',
 'what',
 'very',
 'has',
 'should',
 'yours',
 "weren't",
 'its',
 'is',
 'those',
 "you'll",
 'that',
 'down',
 'o',
 'ourselves',
 'needn',
 'their',
 'ours',
 'above',
 'ain',
 'into',
 "doesn't",
 'before',
 'herself',
 're',
 'out',
 've',
 'when',
 'than',
 "wasn't",
 'same',
 'i',
 'it',
 'used',
 'only',
 'does',
 'an',
 "wouldn't",
 'the',
 'can',
 'aren',
 'which',
 'do',
 'y',
 's',
 'there',
 'are',
 'be',
 'me',
 "didn't",
 'mustn',
 'our',
 'how',
 "hadn't",
 'who',
 'more',
 'against',
 "you're",
 'because',
 'yourselves',
 'if',
 "it's",
 "won't",
 'below',
 'hers',
 'a',
 "hasn't",
 'for',
 'didn',
 'both',
 'further',
 'been',
 'nor',
 'wasn',
 'shouldn',
 'himself',
 'such',
 'themselves',
 'haven',
 'again',
 'about',
 'during',
 'myself',
 'few',
 'ma',
 'theirs',
 'mightn',
 "haven't",
 'hadn',
 'had',
 'don',
 'then',
 'in',
 "shouldn't",
 'being',
 'm',
 'now',
 "should've",
 'doing',
 'her',
 'your',
 'from',
 'with',
 'couldn',
 "couldn't",
 'why']

In [9]:
df_valid = pd.read_csv("/kaggle/input/kaggle-llm-science-exam/test.csv")

In [10]:
retrieved_articles_parsed = get_relevant_documents_parsed(df_valid)
gc.collect()

Map (num_proc=2):   0%|          | 0/2101279 [00:00<?, ? examples/s]

  0%|          | 0/1 [00:00<?, ?it/s]/opt/conda/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:409: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'"] not in stop_words.
  warnings.warn(


length of vectorizer vocab is 20185



100%|██████████| 1/1 [06:40<00:00, 400.12s/it]


18

In [11]:
retrieved_articles = get_relevant_documents(df_valid)
gc.collect()

Map (num_proc=2):   0%|          | 0/2781652 [00:00<?, ? examples/s]

  0%|          | 0/1 [00:00<?, ?it/s]/opt/conda/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:409: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'"] not in stop_words.
  warnings.warn(


length of vectorizer vocab is 20185



100%|██████████| 1/1 [05:12<00:00, 312.18s/it]


18

In [12]:
from transformers import AutoTokenizer
from transformers import AutoModelForMultipleChoice

In [13]:
# tokenizer = LongformerTokenizer.from_pretrained("/kaggle/input/model-v2")
# model = LongformerForMultipleChoice.from_pretrained("/kaggle/input/model-v2").cuda()

model_dir = "/kaggle/input/model-vwhole-dataset-512"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForMultipleChoice.from_pretrained(model_dir).cuda()

In [14]:
# predictions = []
# submit_ids = []

# for index in tqdm(range(df_valid.shape[0])):
#     columns = df_valid.iloc[index].values
#     submit_ids.append(columns[0])
#     question = columns[1]
#     options = [columns[2], columns[3], columns[4], columns[5], columns[6]]
#     context1 = f"{retrieved_articles[index][-4][2]}\n{retrieved_articles[index][-3][2]}\n{retrieved_articles[index][-2][2]}\n{retrieved_articles[index][-1][2]}"
#     context2 = f"{retrieved_articles_parsed[index][-3][2]}\n{retrieved_articles_parsed[index][-2][2]}\n{retrieved_articles_parsed[index][-1][2]}"
#     inputs1 = prepare_answering_input(
#         tokenizer=tokenizer, question=question,
#         options=options, context=context1,
#         )
#     inputs2 = prepare_answering_input(
#         tokenizer=tokenizer, question=question,
#         options=options, context=context2,
#         )
    
#     inputs3 = prepare_answering_input(
#         tokenizer=tokenizer_2, question=question,
#         options=options, context=context1,
#         )
#     inputs4 = prepare_answering_input(
#         tokenizer=tokenizer_2, question=question,
#         options=options, context=context2,
#         )
    
#     with torch.no_grad():
#         outputs1 = model(**inputs1)    
#         losses1 = -outputs1.logits[0].detach().cpu().numpy()
#         probability1 = torch.softmax(torch.tensor(-losses1), dim=-1)
        
#     with torch.no_grad():
#         outputs2 = model(**inputs2)
#         losses2 = -outputs2.logits[0].detach().cpu().numpy()
#         probability2 = torch.softmax(torch.tensor(-losses2), dim=-1)
    
#     with torch.no_grad():
#         outputs3 = model_2(**inputs3)    
#         losses3 = -outputs3.logits[0].detach().cpu().numpy()
#         probability3 = torch.softmax(torch.tensor(-losses3), dim=-1)
        
#     with torch.no_grad():
#         outputs4 = model_2(**inputs4)
#         losses4 = -outputs4.logits[0].detach().cpu().numpy()
#         probability4 = torch.softmax(torch.tensor(-losses4), dim=-1)
            
#     probability_ = (probability1 + probability2 + probability3 + probability4)/4

#     if probability_.max() > 0.3:
#         predict = np.array(list("ABCDE"))[np.argsort(probability_)][-3:].tolist()[::-1]
#     else:
#         predict = backup_model_predictions.iloc[index].prediction.replace(" ","")
#     predictions.append(predict)

# predictions = [" ".join(i) for i in predictions]

In [15]:
# pd.DataFrame({'id':submit_ids,'prediction':predictions}).to_csv('submission.csv', index=False)

In [16]:
DEBUG = False
VAL_SIZE = 200 if DEBUG else 1500

val_df = pd.read_csv('/kaggle/input/mmlu-dataset-valid-only/valid_mmlu_1526_ind0.csv',index_col=0)[:VAL_SIZE]

val_df['E'] = '' # dummy answer that allows us to preprocess the test datataset using functionality that works for the train set
val_df = val_df.replace(np.NaN, '')

val_df['A'] = val_df['A'].map(str)
val_df['B'] = val_df['B'].map(str)
val_df['C'] = val_df['C'].map(str)
val_df['D'] = val_df['D'].map(str)
val_df['E'] = val_df['E'].map(str)

val_df.reset_index(inplace=True, drop=True)

In [17]:
retrieved_articles_new = get_relevant_documents(val_df)
gc.collect()

  0%|          | 0/2 [00:00<?, ?it/s]/opt/conda/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:409: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'"] not in stop_words.
  warnings.warn(


length of vectorizer vocab is 61574



 50%|█████     | 1/2 [06:24<06:24, 384.17s/it]/opt/conda/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:409: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'"] not in stop_words.
  warnings.warn(


length of vectorizer vocab is 66301



100%|██████████| 2/2 [12:32<00:00, 376.07s/it]


0

In [18]:
retrieved_articles_parsed_new = get_relevant_documents_parsed(val_df)
gc.collect()

  0%|          | 0/3 [00:00<?, ?it/s]/opt/conda/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:409: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'"] not in stop_words.
  warnings.warn(


length of vectorizer vocab is 42615



 33%|███▎      | 1/3 [07:16<14:33, 436.80s/it]/opt/conda/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:409: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'"] not in stop_words.
  warnings.warn(


length of vectorizer vocab is 66378



 67%|██████▋   | 2/3 [14:33<07:16, 436.48s/it]/opt/conda/lib/python3.10/site-packages/sklearn/feature_extraction/text.py:409: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ["'"] not in stop_words.
  warnings.warn(


length of vectorizer vocab is 21420



100%|██████████| 3/3 [21:18<00:00, 426.32s/it]


0

In [19]:
probability1 = []
probability2 = []
submit_ids = []

for index in tqdm(range(val_df.shape[0])):
    columns = val_df.iloc[index].values
    submit_ids.append(columns[0])
    question = columns[1]
    options = [columns[2], columns[3], columns[4], columns[5], columns[6]]
    context1 = f"{retrieved_articles_new[index][-4][2]}\n{retrieved_articles_new[index][-3][2]}\n{retrieved_articles_new[index][-2][2]}\n{retrieved_articles_new[index][-1][2]}"
    context2 = f"{retrieved_articles_parsed_new[index][-3][2]}\n{retrieved_articles_parsed_new[index][-2][2]}\n{retrieved_articles_parsed_new[index][-1][2]}"
    inputs1 = prepare_answering_input(
        tokenizer=tokenizer, question=question,
        options=options, context=context1, model=model
        )
    inputs2 = prepare_answering_input(
        tokenizer=tokenizer, question=question,
        options=options, context=context2, model=model
        )
    
    with torch.no_grad():
        outputs1 = model(**inputs1)    
        losses1 = -outputs1.logits[0].detach().cpu().numpy()
        probability1.append(torch.softmax(torch.tensor(-losses1), dim=-1).numpy())
        
    with torch.no_grad():
        outputs2 = model(**inputs2)
        losses2 = -outputs2.logits[0].detach().cpu().numpy()
        probability2.append(torch.softmax(torch.tensor(-losses2), dim=-1).numpy())

 31%|███▏      | 469/1500 [04:31<09:56,  1.73it/s]


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:27                                                                                   │
│                                                                                                  │
│   24 │   │   probability1.append(torch.softmax(torch.tensor(-losses1), dim=-1).numpy())          │
│   25 │                                                                                           │
│   26 │   with torch.no_grad():                                                                   │
│ ❱ 27 │   │   outputs2 = model(**inputs2)                                                         │
│   28 │   │   losses2 = -outputs2.logits[0].detach().cpu().numpy()                                │
│   29 │   │   probability2.append(torch.softmax(torch.tensor(-losses2), dim=-1).numpy())          │
│   30                                                                                             │
│                                                                                                  │
│ /opt/conda/lib/python3.10/site-packages/torch/nn/modules/module.py:1501 in _call_impl            │
│                                                                                                  │
│   1498 │   │   if not (self._backward_hooks or self._backward_pre_hooks or self._forward_hooks   │
│   1499 │   │   │   │   or _global_backward_pre_hooks or _global_backward_hooks                   │
│   1500 │   │   │   │   or _global_forward_hooks or _global_forward_pre_hooks):                   │
│ ❱ 1501 │   │   │   return forward_call(*args, **kwargs)                                          │
│   1502 │   │   # Do not call functions when jit is used                                          │
│   1503 │   │   full_backward_hooks, non_full_backward_hooks = [], []                             │
│   1504 │   │   backward_pre_hooks = []                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.10/site-packages/transformers/models/deberta_v2/modeling_deberta_v2.py:16 │
│ 15 in forward                                                                                    │
│                                                                                                  │
│   1612 │   │   │   else None                                                                     │
│   1613 │   │   )                                                                                 │
│   1614 │   │                                                                                     │
│ ❱ 1615 │   │   outputs = self.deberta(                                                           │
│   1616 │   │   │   flat_input_ids,                                                               │
│   1617 │   │   │   position_ids=flat_position_ids,                                               │
│   1618 │   │   │   token_type_ids=flat_token_type_ids,                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.10/site-packages/torch/nn/modules/module.py:1501 in _call_impl            │
│                                                                                                  │
│   1498 │   │   if not (self._backward_hooks or self._backward_pre_hooks or self._forward_hooks   │
│   1499 │   │   │   │   or _global_backward_pre_hooks or _global_backward_hooks                   │
│   1500 │   │   │   │   or _global_forward_hooks or _global_forward_pre_hooks):                   │
│ ❱ 1501 │   │   │   return forward_call(*args, **kwargs)                                          │
│   1502 │   │   # Do not call functions when jit is used                                          │
│   1503 │   │   full_backward_hooks, non_full_backward_hooks

In [ ]:
del tokenizer, model, outputs1, losses1, outputs2, losses2
gc.collect()

In [ ]:
model_dir_2 = "/kaggle/input/model-v2"
tokenizer_2 = AutoTokenizer.from_pretrained(model_dir_2)
model_2 = AutoModelForMultipleChoice.from_pretrained(model_dir_2).cuda()

In [ ]:
probability3 = []
probability4 = []
submit_ids = []

for index in tqdm(range(val_df.shape[0])):
    columns = val_df.iloc[index].values
    submit_ids.append(columns[0])
    question = columns[1]
    options = [columns[2], columns[3], columns[4], columns[5], columns[6]]
    context1 = f"{retrieved_articles_new[index][-4][2]}\n{retrieved_articles_new[index][-3][2]}\n{retrieved_articles_new[index][-2][2]}\n{retrieved_articles_new[index][-1][2]}"
    context2 = f"{retrieved_articles_parsed_new[index][-3][2]}\n{retrieved_articles_parsed_new[index][-2][2]}\n{retrieved_articles_parsed_new[index][-1][2]}"
    
    inputs3 = prepare_answering_input(
        tokenizer=tokenizer_2, question=question,
        options=options, context=context1, model=model_2
        )
    inputs4 = prepare_answering_input(
        tokenizer=tokenizer_2, question=question,
        options=options, context=context2, model=model_2
        )

    with torch.no_grad():
        outputs3 = model_2(**inputs3)    
        losses3 = -outputs3.logits[0].detach().cpu().numpy()
        probability3.append(torch.softmax(torch.tensor(-losses3), dim=-1).numpy())
        
    with torch.no_grad():
        outputs4 = model_2(**inputs4)
        losses4 = -outputs4.logits[0].detach().cpu().numpy()
        probability4.append(torch.softmax(torch.tensor(-losses4), dim=-1).numpy())

In [ ]:
del tokenizer_2, model_2, outputs3, losses3, outputs4, losses4
gc.collect()

In [ ]:
model_dir_3 = "/kaggle/input/checkpoint-1700-tfidf-30k-context-context2-768"
tokenizer_3 = AutoTokenizer.from_pretrained(model_dir_3)
model_3 = AutoModelForMultipleChoice.from_pretrained(model_dir_3).cuda()

In [ ]:
probability5 = []
probability6 = []
submit_ids = []

for index in tqdm(range(val_df.shape[0])):
    columns = val_df.iloc[index].values
    submit_ids.append(columns[0])
    question = columns[1]
    options = [columns[2], columns[3], columns[4], columns[5], columns[6]]
    context1 = f"{retrieved_articles_new[index][-4][2]}\n{retrieved_articles_new[index][-3][2]}\n{retrieved_articles_new[index][-2][2]}\n{retrieved_articles_new[index][-1][2]}"
    context2 = f"{retrieved_articles_parsed_new[index][-3][2]}\n{retrieved_articles_parsed_new[index][-2][2]}\n{retrieved_articles_parsed_new[index][-1][2]}"
    
    inputs5 = prepare_answering_input(
        tokenizer=tokenizer_3, question=question,
        options=options, context=context1, model=model_3
        )
    inputs6 = prepare_answering_input(
        tokenizer=tokenizer_3, question=question, model=model_3
        options=options, context=context2,
        )
    
    with torch.no_grad():
        outputs5 = model_3(**inputs5)    
        losses5 = -outputs5.logits[0].detach().cpu().numpy()
        probability5.append(torch.softmax(torch.tensor(-losses5), dim=-1).numpy())
        
    with torch.no_grad():
        outputs6 = model_3(**inputs6)
        losses6 = -outputs6.logits[0].detach().cpu().numpy()
        probability6.append(torch.softmax(torch.tensor(-losses6), dim=-1).numpy())

In [ ]:
del tokenizer_3, model_3, outputs5, losses5, outputs6, losses6
gc.collect()

In [ ]:
prob_lables = ['A_prob', 'B_prob', 'C_prob', 'D_prob', 'E_prob']
probability1_res = pd.DataFrame(probability1, columns=prob_lables)
probability2_res = pd.DataFrame(probability2, columns=prob_lables)
probability3_res = pd.DataFrame(probability3, columns=prob_lables)
probability4_res = pd.DataFrame(probability4, columns=prob_lables)
probability5_res = pd.DataFrame(probability5, columns=prob_lables)
probability6_res = pd.DataFrame(probability6, columns=prob_lables)

In [ ]:
probability1_res

### Optimise model weights

In [ ]:
from scipy.optimize import minimize, fsolve
import datetime
import torch.nn.functional as F
from numba import njit

In [ ]:
def apk(actual, predicted, k=5):
    """
    Computes the average precision at k.
    This function computes the average prescision at k between two lists of
    items.
    Parameters
    ----------
    actual : list
             A list of elements that are to be predicted (order doesn't matter)
    predicted : list
                A list of predicted elements (order does matter)
    k : int, optional
        The maximum number of predicted elements
    Returns
    -------
    score : double
            The average precision at k over the input lists
    """
    
    # requires all elements are unique
    assert (len(np.unique(predicted)) == len(predicted))

    if len(predicted)>k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i,p in enumerate(predicted):
        # first condition checks whether it is valid prediction
        # second condition checks if prediction is not repeated
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)

    return score / min(len(actual), k)


def mapk(actual, predicted, k=5):
    
    """
    Computes the mean average precision at k.
    This function computes the mean average prescision at k between two lists
    of lists of items.
    Parameters
    ----------
    actual : list
             A list of lists of elements that are to be predicted 
             (order doesn't matter in the lists)
    predicted : list
                A list of lists of predicted elements
                (order matters in the lists)
    k : int, optional
        The maximum number of predicted elements
    Returns
    -------
    score : double
            The mean average precision at k over the input lists
    """
    return np.mean([apk(a,p,k) for a,p in zip(actual, predicted)])

In [ ]:
@njit
def grad_func_jit(weights):
    preds_clip = np.minimum(1 - 1e-15, np.maximum(preds, 1e-15))
    gradients = np.zeros(preds.shape[0])
    for i in range(preds.shape[0]):
        a, b, c = target_values, preds_clip[i], np.zeros((preds.shape[1], preds.shape[2]))
        a = np.eye(5)[a]
        for j in range(preds.shape[0]):
            if j != i:
                c += weights[j] * preds_clip[j]
        gradients[i] = -np.mean((-a*b+(b**2)*weights[i]+b*c)/((b**2)*(weights[i]**2)+2*b*c*weights[i]-b*weights[i]+(c**2)-c))
    return gradients

In [ ]:
def calc_mtr(predicted, k=3):
    y_preds = np.argsort(-predicted, 1)
    map3 = mapk(target_values.reshape(-1, 1), y_preds.reshape(-1, 5), k=k)
    return map3

# def calc_loss(predicted):
#     score = F.cross_entropy(torch.tensor(predicted), torch.tensor(target_values)).numpy()
#     return score


def log_loss_numpy(y_pred):
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    loss = - target_values_one_hot * np.log(y_pred)
    loss = np.sum(loss, axis=-1)
    return loss.mean()

def func_to_optimise(weights):
    pred_blend = np.tensordot(weights, preds, axes = ((0), (0)))
    score = log_loss_numpy(pred_blend)
    return score

def func_to_map3(weights):
    pred_blend = np.tensordot(weights, preds, axes = ((0), (0)))
    score = calc_mtr(pred_blend)
    return score


In [ ]:
options = 'ABCDE'
indices = list(range(5))

option_to_index = {option: index for option, index in zip(options, indices)}
index_to_option = {index: option for option, index in zip(options, indices)}
target_values = val_df['answer'].map(option_to_index).values
target_values_one_hot = np.eye(5)[target_values]

In [ ]:
preds_dict = {
    'model-512-1': probability1_res,
    'model-512-2': probability2_res,
    'model-v2-1': probability3_res,   
    'model-v2-2': probability4_res,
    'model-768-1': probability5_res,   
    'model-768-2': probability6_res,
}


In [ ]:
probability1_res

In [ ]:
preds = np.zeros((len(preds_dict), len(val_df), 5))
for i in range(preds.shape[0]):
    preds[i] = list(preds_dict.values())[i]

In [ ]:
%%time

map3_scores = {}
for n, key in enumerate(preds_dict.keys()):
    score_val = calc_mtr(preds[n])
    map3_scores[key] = score_val
    print(f'{key:40s} CV_map@3:', score_val)
    
print('-' * 60)

loss_scores = {}
for n, key in enumerate(preds_dict.keys()):
    score_val = log_loss_numpy(preds[n])
    loss_scores[key] = score_val
    print(f'{key:40s} CV_CELoss:', score_val)
    
print('-' * 60)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.style as style
import seaborn as sns
from matplotlib import pyplot
from matplotlib.ticker import ScalarFormatter
sns.set_context("talk")
style.use('fivethirtyeight')

subs = np.zeros((len(preds_dict), len(val_df), 5))

for i, p in enumerate(preds_dict.keys()):
    print(i,p)
    subs[i,:,:] = list(preds_dict.values())[i]
    
corr = np.corrcoef(subs.reshape(len(preds_dict), -1))

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(15, 12))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(220, 10, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr, cmap=cmap, annot=True, fmt="g",
            square=True, linewidths=.5, cbar_kws={"shrink": .5}, ax=ax)
ax.set_ylim(corr.shape[0], 0)
plt.yticks(rotation=0)

In [ ]:
import time

tol = 1e-10
init_guess = [1 / preds.shape[0]] * preds.shape[0]
bnds = [(0, 1) for _ in range(preds.shape[0])]
cons = {'type': 'eq', 
        'fun': lambda x: np.sum(x) - 1, 
        'jac': lambda x: [1] * len(x)}

print('Inital Blend Loss:', func_to_optimise(init_guess))
print('Inital Blend MAP@3:', func_to_map3(init_guess))
start_time = time.time()

res_scipy = minimize(fun = func_to_optimise, 
                     x0 = init_guess, 
                     method = 'SLSQP', 
                     tol = tol,
                     bounds = bnds,
                     jac = grad_func_jit, 
                     constraints = cons,
                     options={"disp":True,"maxiter":1000})

print(f'[{str(datetime.timedelta(seconds = time.time() - start_time))[2:7]}] Optimised Blend Loss:', res_scipy.fun, ', Optimised Blend MAP@3:', func_to_map3(res_scipy.x))
print('Optimised Weights:', res_scipy.x)
print('-' * 70)

for n, key in enumerate(preds_dict.keys()):
    print(f'{key:40s} Optimised Weights:', res_scipy.x[n])

In [ ]:
ws = [res_scipy.x[i] for i in range(len(preds_dict.keys()))]
ws = ws / np.sum(ws)
ws

In [ ]:
print(ws)

In [ ]:
# probability1_new = []
# probability2_new = []
# probability3_new = []
# probability4_new = []
# probability5_new = []
# probability6_new = []

# predictions_res = []
# submit_ids = []

# for index in tqdm(range(df_valid.shape[0])):
#     columns = df_valid.iloc[index].values
#     submit_ids.append(columns[0])
#     question = columns[1]
#     options = [columns[2], columns[3], columns[4], columns[5], columns[6]]
#     context1 = f"{retrieved_articles[index][-4][2]}\n{retrieved_articles[index][-3][2]}\n{retrieved_articles[index][-2][2]}\n{retrieved_articles[index][-1][2]}"
#     context2 = f"{retrieved_articles_parsed[index][-3][2]}\n{retrieved_articles_parsed[index][-2][2]}\n{retrieved_articles_parsed[index][-1][2]}"
#     inputs1 = prepare_answering_input(
#         tokenizer=tokenizer, question=question,
#         options=options, context=context1,
#         )
#     inputs2 = prepare_answering_input(
#         tokenizer=tokenizer, question=question,
#         options=options, context=context2,
#         )
    
#     inputs3 = prepare_answering_input(
#         tokenizer=tokenizer_2, question=question,
#         options=options, context=context1,
#         )
#     inputs4 = prepare_answering_input(
#         tokenizer=tokenizer_2, question=question,
#         options=options, context=context2,
#         )
    
#     inputs5 = prepare_answering_input(
#         tokenizer=tokenizer_3, question=question,
#         options=options, context=context1,
#         )
#     inputs6 = prepare_answering_input(
#         tokenizer=tokenizer_3, question=question,
#         options=options, context=context2,
#         )
    
#     with torch.no_grad():
#         outputs1 = model(**inputs1)    
#         losses1 = -outputs1.logits[0].detach().cpu().numpy()
#         probability1_new = torch.softmax(torch.tensor(-losses1), dim=-1)
        
#     with torch.no_grad():
#         outputs2 = model(**inputs2)
#         losses2 = -outputs2.logits[0].detach().cpu().numpy()
#         probability2_new = torch.softmax(torch.tensor(-losses2), dim=-1)
    
#     with torch.no_grad():
#         outputs3 = model_2(**inputs3)    
#         losses3 = -outputs3.logits[0].detach().cpu().numpy()
#         probability3_new = torch.softmax(torch.tensor(-losses3), dim=-1)
        
#     with torch.no_grad():
#         outputs4 = model_2(**inputs4)
#         losses4 = -outputs4.logits[0].detach().cpu().numpy()
#         probability4_new = torch.softmax(torch.tensor(-losses4), dim=-1)
        
#     with torch.no_grad():
#         outputs5 = model_3(**inputs5)    
#         losses5 = -outputs5.logits[0].detach().cpu().numpy()
#         probability5_new = torch.softmax(torch.tensor(-losses5), dim=-1).numpy()
        
#     with torch.no_grad():
#         outputs6 = model_3(**inputs6)
#         losses6 = -outputs6.logits[0].detach().cpu().numpy()
#         probability6_new = torch.softmax(torch.tensor(-losses6), dim=-1).numpy()
            
#     probability_ = probability1_new * ws[0] + probability2_new * ws[1] + probability3_new * ws[2] + probability4_new * ws[3] + probability5_new * ws[4] + probability6_new * ws[5]

#     if probability_.max() > 0.3:
#         predict = np.array(list("ABCDE"))[np.argsort(probability_)][-3:].tolist()[::-1]
#     else:
#         predict = backup_model_predictions.iloc[index].prediction.replace(" ","")
#     predictions_res.append(predict)

# predictions = [" ".join(i) for i in predictions_res]

In [ ]:
# pd.DataFrame({'id':submit_ids,'prediction':predictions}).to_csv('submission.csv', index=False)